# PV Systematic v4 — Full Pipeline + ZoLa + QA + Auto-Export
Run Stages 1–10.

### Imports and Config


In [13]:
import os
import requests
import geopandas as gpd
import pandas as pd
import folium
from shapely.geometry import Polygon, box
import subprocess
from folium.plugins import MarkerCluster
#version im fucking up why did this take so nlong the firdt timr

## Stage 1: Define Bounding Box, Fetch + Save Footprints


In [ ]:
#Bounding box coordinates 
N_LAT = 40.7358876957036
S_LAT = 40.71195687391828
W_LON = -74.00188877257987
E_LON = -73.9721415592451

# NYC building footprints? 
FOOTPRINTS_URL = "https://data.cityofnewyork.us/resource/5zhs-2jue.geojson" #ok so address doesnt exist here, therefore DNE in footprints.geojson  therefore DNE in gdf. Doesnt get preserved in the pv_from_footprints.py script. We have to get it from somewhere else, maybe from the all_walkups_6story.csv file that has the address and bbl and then merge it with the footprints data using the bbl as the key? 

#Search Parameters 
params = {
    "$select": "*",
    "$where": f"within_box(the_geom,{S_LAT},{W_LON},{N_LAT},{E_LON})",
    "$limit": 50000,
}


# This is where the API is being called to fetch the footprints data for the specified bounding box (we dont create a variable for the bounding box until further down). The parameters include a spatial query to filter the data within the defined box and a limit on the number of records returned.
r = requests.get(FOOTPRINTS_URL, params=params)
r.raise_for_status()

#Downloads all footprint polygons inside the AOI and saves them as:
#so why do we have to first save this as geojson and read it in pandas to manipulate it?
with open("footprints.geojson", "wb") as f:
    f.write(r.content)

# Geopandas reads the GeoJSON file with the footprints. GDF now contains all the geometries and attributes of the building footprints, which can be used for further analysis or visualization. One of the most important variables. 
gdf = gpd.read_file("footprints.geojson")

# normalize BBL
gdf = gdf.rename(columns={"base_bbl": "bbl"})
gdf["bbl"] = gdf["bbl"].astype(str).str.zfill(10)


# does gdf have address? 
print("gdf has address:", "address" in gdf.columns)

print("Footprints:", len(gdf))


gdf has address: False
Footprints: 7384


### Stage 2: Generate PV Data from Footprints

In [15]:
#This launches the solar estimation script externally. That script reads footprints.geojson and generates all PV outputs.
subprocess.run(["python", "pv_from_footprints.py"], check=True)

PV artifacts written.


CompletedProcess(args=['python', 'pv_from_footprints.py'], returncode=0)

### Stage 3: Load PV Footprints and Clip to AOI Polygon 


In [16]:
# #Loading the calculated solar data "pv_canopy_footprints.geojson" and "pv_canopy_centroids.geojson" created in footprints.py, right?  
#  Reading in the geojson as a geodataframe. The pv_from_footprints.py script is responsible for creating these two geojson files with the solar estimation data, and here we are loading them into GeoDataFrames for further analysis and manipulation. 

pv_fp = gpd.read_file("pv_canopy_footprints.geojson")
pv_centroids = gpd.read_file("pv_canopy_centroids.geojson")

# Clean data by selecting only the relevant columns for the PV footprints. This step is crucial for focusing on the key attributes needed for analysis and visualization, such as the building identifier (bbl), the estimated DC power of the PV system (pv_kw_dc), the annual energy production (annual_kwh), and the geometry of the PV canopy. By filtering out unnecessary columns, we can streamline our data processing and make it easier to work with the essential information. 
pv_fp = pv_fp[[
    "bbl",
    "pv_kw_dc",
    "annual_kwh",
    "geometry", 
    "address"
]]
#The address most likely originated from: all_walkups_6story.csv or MapPLUTO then passed through pv_from_footprints.py



# Defining polygon coordinates
polygon_coords = [
    (-73.99075, 40.73474), #A: 14th & Broadway 
    ( -73.97209, 40.72689), #B: 14th and FDR 
    (-73.9777,  40.71314), #C: Grand & FDR 
    (-73.98249,  40.71468), #D: Grand and E Broadway  
    (-73.99411, 40.71371), # E : E broadway and Manhattan Bridge  
    (-73.99484, 40.7158), #F: Canal and Manhattan bridge
    (-73.9986, 40.71711),  # G: Canal and Mulberry  
    (-74.00187, 40.71939),  # H: Canal and Broadway  
    (-73.99143, 40.73179),  # I: Broadway and 10th 
     (-73.99075, 40.73474)  #A: 14th & Broadway, close polygon
]


#Using the Shapely library, Create polygon for clipping.
# GeoDataFrame is created to hold the polygon geometry with the appropriate coordinate reference system (CRS). 
# is gdf employed becuse it is easier to manipulate this specific thing with it rather than "raw" geopandas? yes. these libraries help us clean up/prep data and format it the way we want so we can easily access index use it. this block i need to study to see if we really need bounding box or not. and the script.
poly = Polygon(polygon_coords)
poly_gdf = gpd.GeoDataFrame(geometry=[poly], crs="EPSG:4326")


# Clip every layer to the Polygon AOI
pv_fp_poly = gpd.clip(pv_fp, poly_gdf)
buildings_poly = gpd.clip(gdf, poly_gdf)
pv_centroids_poly = gpd.clip(pv_centroids, poly_gdf)

# Creating a stored bounding box for debugging  
bbox_geom = box(W_LON, S_LAT, E_LON, N_LAT)
bbox_gdf = gpd.GeoDataFrame(geometry=[bbox_geom], crs="EPSG:4326")

# clean data.  does this need to be at the end 
# pv_fp = pv_fp[[
#     "bbl",
#     "pv_kw_dc",
#     "annual_kwh",
#     "geometry"
# ]]

### Stage 4: Data Cleaning For Folium

In [ ]:
#Buildings_poly originally only has footprint geometry + BBL. pv_fp_poly already contains an address field. The merge copies address into the footprint layer.
buildings_poly = buildings_poly.merge(
    pv_fp_poly[["bbl", "address"]].drop_duplicates(),
    on="bbl",
    how="left"
)
print(buildings_poly.columns)

# Cleaning each layer's data for folium by converting every non-geometry field to string. Folium sometimes crashes when GeoJSON properties contain NumPy value, mixed types, nulls, non-serializable objects. This ensures the map can safely render.
def make_json_safe(gdf):
    gdf = gdf.copy()
    for col in gdf.columns:
        if col != "geometry":
            gdf[col] = gdf[col].astype(str)
    return gdf

bbox_safe = make_json_safe(bbox_gdf) #bounding box for debugging 
poly_safe = make_json_safe(poly_gdf) #polygon AOI for debugging 
buildings_safe = make_json_safe(buildings_poly) #building footprints clipped to polygon AOI 
pv_fp_poly_safe = make_json_safe(pv_fp_poly)
pv_centroids_poly_safe = make_json_safe(pv_centroids_poly)


# Debugging outputs to verify data integrity and understand what attributes are available for tooltips and visualization. This helps ensure that the expected fields are present after clipping and cleaning, and that the data is ready for mapping with Folium.

print("bounding box aoi COLUMNS:", bbox_safe.columns.tolist())
print("polygon aio COLUMNS:", poly_safe.columns.tolist())
print("buildings_poly has address:", "address" in buildings_poly.columns)
print("buildings_safe has address:", "address" in buildings_safe.columns)

print("poly_safe footprints COLUMNS:", poly_safe.columns.tolist())
print("pv footprintspoly (safe) COLUMNS:", pv_fp_poly_safe.columns.tolist())



bounding box aoi COLUMNS: ['geometry']
polygon aio COLUMNS: ['geometry']
buildings_poly has address: False
buildings_safe has address: False
poly_safe footprints COLUMNS: ['geometry']
pv footprintspoly (safe) COLUMNS: ['bbl', 'pv_kw_dc', 'annual_kwh', 'geometry', 'address']


### Stage 5: Build Folium Map

In [18]:

#OpenStreetMap tile servers rejecting your requests and causing html blockages/popups. Trying CartoDB 
# m = folium.Map(location=[(N_LAT+S_LAT)/2, (E_LON+W_LON)/2], zoom_start=14)

m = folium.Map( 
    location=[N_LAT, E_LON],
    zoom_start=14,
    tiles="CartoDB positron"
)


def safe_tooltip_fields(gdf, desired_fields):
    return [f for f in desired_fields if f in gdf.columns]


fields = [
    "address", "class", "name", "bbl", "shape_area",
    "construction_year", "ground_elevation",
    "mappluto_bbl", "objectid", "feature_code",
    "shape_length", "height_roof",
    "last_status_type", "bin"
]

b_fields = safe_tooltip_fields(buildings_safe, fields)
pv_fields = safe_tooltip_fields(pv_fp_poly_safe, fields)

solar_fields = [
    "pv_area",
    "pv_kw_dc",
    "psh_daily_kwh",
    "annual_kwh"
]

solar_fields = safe_tooltip_fields(pv_fp_poly_safe, solar_fields)
combined_fields = pv_fields + solar_fields

combined_aliases = (
    [f"Building: {f}" for f in pv_fields] +
    [f"Solar: {f}" for f in solar_fields]
)


# --- Bounding Box for debugging ---
folium.GeoJson(
    bbox_safe,
    name="Bounding Box Perimeter",
    style_function=lambda x: {
        "color": "blue",
        "weight": 3,
        "fillOpacity": 0
    }
).add_to(m)



# --- Polygon AOI for debugging ---
folium.GeoJson(
    poly_safe,
    name="Polygon AOI Perimeter",
    style_function=lambda x: {
        "color": "purple",
        "weight": 3,
        "fillOpacity": 0
    }
).add_to(m)



# --- Individual Building Footprints in AOI ---
folium.GeoJson(
    buildings_safe,
    name="Building Footprints (All)",
    tooltip=folium.GeoJsonTooltip (
    fields=b_fields,
    aliases=[f"{f}: " for f in b_fields],
    sticky=True
    ),
    style_function=lambda x: {
        "fill": "#9ecae1",
        "stroke": "#6baed6",
        "weight": 1,
        "fillOpacity": 0.2
    }
).add_to(m)



# --- Individual PV Footprints  ---
folium.GeoJson(
    pv_fp_poly_safe,
    name="PV Canopy Footprints",
     tooltip=folium.GeoJsonTooltip(
        fields=combined_fields,
        aliases=combined_aliases,
        sticky=True 
    ),
    style_function=lambda x: {
         "fillColor": "#ffb347",
         "color": "#f59f00",
         "weight": 1,
         "fillOpacity": 0.6
    }
).add_to(m)


# --- PV Centroids (clustered + toggleable) ---
cluster_group = folium.FeatureGroup(name="PV Sites (Centroids)", show=True)
cluster = MarkerCluster().add_to(cluster_group)

for r in pv_centroids_poly.itertuples(index=False):
    lat, lon = r.geometry.y, r.geometry.x

    addr = getattr(r, "address", None) or "(address unavailable)"
    bbl  = getattr(r, "bbl", None) or "(BBL unavailable)"
    pvkw = getattr(r, "pv_kw_dc", None)
    pshs = getattr(r, "psh_daily_kwh", None)

    pvkw_txt = f"{float(pvkw):.1f} kW" if pvkw not in [None, "nan"] else "n/a"
    psh_txt  = f"{float(pshs):.2f} PSH" if pshs not in [None, "nan"] else "n/a"

#if we want clickable popup, use this html block and add popup=folium.Popup(html, max_width=320) to the Marker below.
    # html = (
    #     f"<b>Address:</b> {addr}<br/>"
    #     f"<b>BBL:</b> {bbl}<br/>"
    #     f"<b>PV size:</b> {pvkw_txt}<br/>"
    #     f"<b>PSH:</b> {psh_txt}"
    # )

    annual = getattr(r, "annual_kwh", None)

    annual_txt = f"{float(annual):,.0f} kWh" if annual not in [None, "nan"] else "n/a"
    
    folium.Marker(
        [lat, lon],
        tooltip=f"""
            <b>Solar Data</b><br>
            PV Size: {pvkw_txt}<br>
            PSH: {psh_txt}<br>
            Annual Energy: {annual_txt}
            """,
    ).add_to(cluster)

cluster_group.add_to(m)




# --- Layer control + save ---
folium.LayerControl().add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

import os
os.makedirs("maps", exist_ok=True)
OUT_HTML = "maps/solar_map.html"


m.save(OUT_HTML)
print("Saved", OUT_HTML)

Saved maps/solar_map.html
